# 01 — Frozen backbone feature smoke test
Connect this notebook to a Colab GPU runtime and run from the top. It bootstraps a fresh kernel and independently tests both frozen backbones on one COD image.

In [ ]:
# Fresh-kernel bootstrap. Edit only these settings; cached Drive assets are reused.
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
ACCEPT_COD10K_NONCOMMERCIAL_LICENSE = True

from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, torch

project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev,notebooks]'], check=True)

bootstrap_env = os.environ.copy()
dino_weights = Path('/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth')
if not dino_weights.is_file():
    print('DINOv3 requires approved Meta access on the first run only.')
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
command = [sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
           '--project-dir', str(project_dir), '--state-file', str(state_file)]
subprocess.run(command, cwd=project_dir, env=bootstrap_env, check=True)
bootstrap_env.pop('COD_SSL_DINOV3_DOWNLOAD_URL', None)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = state['project_dir']
DATA_ROOT = state['data_root']
RUNS_ROOT = state['runs_root']
COMPARISONS_ROOT = state['comparisons_root']
SAMPLE_IMAGE = state['sample_image']
TRAIN_MANIFEST = state['train_manifest']
print('Ready on', state['gpu'])


In [22]:
# DINOv3 layers 2, 5, 8, 11
import subprocess, sys
subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/inspect_backbone.py','--backbone','dinov3_vitb16','--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,check=True)

Downloading: "file:///content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vitb16.pth

  0%|          | 0.00/327M [00:00<?, ?B/s]
 34%|███▍      | 112M/327M [00:00<00:00, 1.18GB/s]
 69%|██████▉   | 226M/327M [00:00<00:00, 1.19GB/s]
100%|██████████| 327M/327M [00:00<00:00, 1.19GB/s]
{
  "model": "dinov3_vitb16",
  "repo_commit": "6876159a11b4df116f30f667f8c9888617df0751",
  "checkpoint": "/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth",
  "checkpoint_sha256": "73cec8be7427c8655ceced13ce62f6e20a1fa90d1b4d4a550df17a1144081a7c",
  "total_parameters": 85669632,
  "trainable_parameters": 0,
  "input_shape": [
    1,
    3,
    384,
    384
  ],
  "feature_shapes": [
    [
      1,
      768,
      24,
      24
    ],
    [
      1,
      768,
      24,
      24
    ],
    [
      1,
      768,
      24,
      24
    ],
    [
      1,
      768,
      24,
      24
    ]
  ],
  "dtype": "torch.float32",
  "mean_forward_ms": 265

In [23]:
# Release allocator state before the second process.
import gc
gc.collect(); torch.cuda.empty_cache()

In [24]:
# V-JEPA 2.1 native one-frame pathway; predictor unused.
subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/inspect_backbone.py','--backbone','vjepa21_vitb16','--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,check=True)

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/lib/python3.13/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
{
  "model": "vjepa21_vitb16",
  "repo_commit": "204698b45b3712590f06245fbfba32d3be539812",
  "checkpoint": "/content/drive/MyDrive/cod-ssl/checkpoints/vjepa2_1_vitb_dist_vitG_384.pt",
  "checkpoint_sha256": "848a77c33cc9e6649ed2119c9bea1e2c569bcdab9539ff3e7c02ccc2959ddf4d",
  "total_parameters": 86833152,
  "trainable_parameters": 0,
  "input_shape": [
    1,
    3,
    384,
    384
  ],
  "feature_shapes": [
    [
      1,
      7

Each JSON report must show four feature maps with spatial size `24 × 24` and zero trainable backbone parameters.